#  Product Similarity Engine (SBERT)

Imports

In [47]:
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer, util

Load Data

In [48]:
reviews_df = pd.read_csv('../data/processed/cleaned_reviews.csv')
products_df = pd.read_csv('../data/processed/products.csv')

In [49]:
print("Reviews shape:", reviews_df.shape)
print("Products shape:", products_df.shape)

Reviews shape: (326447, 5)
Products shape: (45911, 3)


# Keep Top Products : performance optimization

In [50]:
top_products = reviews_df['productId'].value_counts().head(200).index
filtered_df = reviews_df[reviews_df['productId'].isin(top_products)]

# Text Representation

In [51]:
filtered_df = filtered_df.drop_duplicates(subset=['text'])
grouped = (
    filtered_df
    .groupby('productId')['text']
    .apply(lambda x: " ".join(x.sample(min(len(x), 5), random_state=42).astype(str)))
    .reset_index()
)
grouped = grouped.merge(products_df[['productId', 'avg_rating', 'rating_count']], on='productId', how='left')

grouped['text'] = (
    "Product ID: " + grouped['productId'].astype(str) + 
    " Rating: " + grouped['avg_rating'].astype(str) + 
    " Count: " + grouped['rating_count'].astype(str) + 
    " Reviews: " + grouped['text'].astype(str)
)
grouped = grouped[['productId', 'text']]
print("Sample:")
print(grouped['text'].iloc[0][:200] + "...")

Sample:
Product ID: B0007A0AQW Rating: 4.542654028436019 Count: 422 Reviews: My Wheaten puppy likes these treats and I like that they're made from good ingredients. But I gave this product three stars, becaus...


Load SBERT Model

In [52]:
model = SentenceTransformer('all-MiniLM-L6-v2')

 Generate Product Embeddings

In [53]:
product_embeddings = model.encode(
    grouped['text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True
)

Batches: 100%|██████████| 3/3 [00:02<00:00,  1.08it/s]


# product : PRODUCT SIMILARITY

In [54]:
product_sim_matrix = util.cos_sim(product_embeddings, product_embeddings).cpu().numpy()

pp_results = []
top_k = 5 
for i in range(len(grouped)):
    scores = product_sim_matrix[i].copy()
    scores[i] = -1  
    top_idx = np.argsort(scores)[-top_k:][::-1]
    
    for idx in top_idx:
        score = round(float(scores[idx]), 4)
        pp_results.append({
            'type': 'product_product',
            'source_productId': grouped.iloc[i]['productId'],
            'similar_productId': grouped.iloc[idx]['productId'],
            'similarity_score': score
        })

pp_df = pd.DataFrame(pp_results)

print("Product-Product similarities:", pp_df.shape)
print(pp_df.head(3))

Product-Product similarities: (325, 4)
              type source_productId similar_productId  similarity_score
0  product_product       B0007A0AQW        B0045XE32E            0.6237
1  product_product       B0007A0AQW        B002LANN56            0.5341
2  product_product       B0007A0AQW        B002QWP89S            0.5246


# REVIEW - PRODUCT SIMILARITY

In [55]:
sample_reviews = (
    filtered_df
    .dropna(subset=['text'])
    .sample(500, random_state=42)
    .reset_index(drop=True)
)

review_embeddings = model.encode(
    sample_reviews['text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True
)

rp_sim_matrix = util.cos_sim(review_embeddings, product_embeddings).cpu().numpy()

rp_results = []

for i in range(len(sample_reviews)):
    top_idx = np.argmax(rp_sim_matrix[i])
    
    rp_results.append({
        'type': 'review_product',
        'source_productId': sample_reviews.iloc[i]['productId'],
        'similar_productId': grouped.iloc[top_idx]['productId'],
        'similarity_score': round(float(rp_sim_matrix[i][top_idx]), 4)
    })

rp_df = pd.DataFrame(rp_results)

print("Review-Product similarities:", rp_df.shape)
print(rp_df.head(3))

Batches: 100%|██████████| 16/16 [00:11<00:00,  1.37it/s]

Review-Product similarities: (500, 4)
             type source_productId similar_productId  similarity_score
0  review_product       B001VJ0B0I        B001VJ0B0I            0.6071
1  review_product       B003VXFK44        B0090X8IPM            0.6513
2  review_product       B005VOONKI        B007TJGZ54            0.6514


In [56]:
final_df = pd.concat([pp_df, rp_df], ignore_index=True)

SAVE OUTPUTS

In [ ]:
final_df.to_csv('../output/task2_similarity.csv', index=False)
np.save('../output/product_similarity_matrix.npy', product_sim_matrix)

final check

In [59]:
print("Saved! Total rows:", len(final_df))
print("\nType distribution:")
print(final_df['type'].value_counts())

print("\nSample output:")
print(final_df.head())

Saved! Total rows: 825

Type distribution:
type
review_product     500
product_product    325
Name: count, dtype: int64

Sample output:
              type source_productId similar_productId  similarity_score
0  product_product       B0007A0AQW        B0045XE32E            0.6237
1  product_product       B0007A0AQW        B002LANN56            0.5341
2  product_product       B0007A0AQW        B002QWP89S            0.5246
3  product_product       B0007A0AQW        B0012NUVN0            0.5230
4  product_product       B0007A0AQW        B0051COPH6            0.4500
